In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv('/Users/amanmujawar/Documents/Projects/team4_project/HR-Analysis/Data/ACSEEO5Y2018.EEOALL1R-2026-02-16T043140.csv', header=0, nrows=3)
print("Shape:", df.shape)
print("Loaded successfully!")

Shape: (3, 20081)
Loaded successfully!


In [3]:
# Full load + structure inspection
df_raw = pd.read_csv('/Users/amanmujawar/Documents/Projects/team4_project/HR-Analysis/Data/ACSEEO5Y2018.EEOALL1R-2026-02-16T043140.csv', header=0, low_memory=False)

print("Full shape:", df_raw.shape)
print("\n--- First column (row labels) ---")
print(df_raw.iloc[:, 0].values)

print("\n--- Sample column names (first 20) ---")
for col in df_raw.columns[:20]:
    print(col)

Full shape: (9, 20081)

--- First column (row labels) ---
['Total, both sexes' '\xa0\xa0\xa0\xa0Number' '\xa0\xa0\xa0\xa0Percent'
 '\xa0\xa0\xa0\xa0Male' '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0Number'
 '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0Percent' '\xa0\xa0\xa0\xa0Female'
 '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0Number'
 '\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0Percent']

--- Sample column names (first 20) ---
Label (Grouping)
Alabama!!Top executives!!Total, race and ethnicity!!Estimate
Alabama!!Top executives!!Total, race and ethnicity!!Margin of Error
Alabama!!Top executives!!Hispanic or Latino!!Estimate
Alabama!!Top executives!!Hispanic or Latino!!Margin of Error
Alabama!!Top executives!!Not Hispanic or Latino, one race!!White alone!!Estimate
Alabama!!Top executives!!Not Hispanic or Latino, one race!!White alone!!Margin of Error
Alabama!!Top executives!!Not Hispanic or Latino, one race!!Black or African American alone!!Estimate
Alabama!!Top executives!!Not Hispanic or Latino, one race!!Black or African Amer

In [ ]:
import pandas as pd
import numpy as np

#  1. Load 
df_raw = pd.read_csv(
    '/Users/amanmujawar/Documents/Projects/team4_project/HR-Analysis/Data/ACSEEO5Y2018.EEOALL1R-2026-02-16T043140.csv',
    header=0, low_memory=False
)

#  2. Keep only Estimate columns (drop Margin of Error) 
estimate_cols = [c for c in df_raw.columns if c.endswith('!!Estimate') or c == 'Label (Grouping)']
df = df_raw[estimate_cols].copy()

#  3. Keep only the "Total Percent" row (row index 2) 
# Row 0 = Total both sexes (label), Row 1 = Number, Row 2 = Percent
df = df.iloc[[2]].copy()

#  4. Melt wide → long 
df_melted = df.melt(id_vars='Label (Grouping)', var_name='column_key', value_name='percent')

#  5. Parse the column key: State !! Occupation !! Race !! Estimate 
split = df_melted['column_key'].str.replace('!!Estimate', '', regex=False).str.split('!!', expand=True)
df_melted['state']      = split[0]
df_melted['occupation'] = split[1]
df_melted['race']       = split[2]

#  6. Clean up percent values 
df_melted['percent'] = (
    df_melted['percent']
    .astype(str)
    .str.replace('%', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df_melted['percent'] = pd.to_numeric(df_melted['percent'], errors='coerce')

#  7. Drop nulls and total rows 
df_clean = df_melted.dropna(subset=['percent', 'race', 'occupation']).copy()
df_clean = df_clean[df_clean['race'] != 'Total, race and ethnicity']

# 8. Drop MOE rows, keep only meaningful race categories ──────────────────
keep_races = [
    'Hispanic or Latino',
    'Not Hispanic or Latino, one race!!White alone',
    'Not Hispanic or Latino, one race!!Black or African American alone',
    'Not Hispanic or Latino, one race!!Asian alone',
    'Not Hispanic or Latino, one race!!American Indian and Alaska Native alone',
]
df_clean = df_clean[df_clean['race'].isin(keep_races)]

# ── 9. Simplify race labels 
race_map = {
    'Hispanic or Latino': 'Hispanic or Latino',
    'Not Hispanic or Latino, one race!!White alone': 'White',
    'Not Hispanic or Latino, one race!!Black or African American alone': 'Black or African American',
    'Not Hispanic or Latino, one race!!Asian alone': 'Asian',
    'Not Hispanic or Latino, one race!!American Indian and Alaska Native alone': 'American Indian / Alaska Native',
}
df_clean['race'] = df_clean['race'].map(race_map)

# ── 10. Filter STEM occupations only 
stem_keywords = [
    'engineer', 'computer', 'software', 'mathematician', 'statistician',
    'scientist', 'architect', 'technician', 'drafter', 'surveyor', 'physicist',
    'chemist', 'biologist', 'life scientist', 'physical scientist'
]
mask = df_clean['occupation'].str.lower().str.contains('|'.join(stem_keywords), na=False)
df_stem = df_clean[mask].copy()

# ── 11. Final cleanup 
df_stem = df_stem[['state', 'occupation', 'race', 'percent']].reset_index(drop=True)

print(" Clean STEM dataframe shape:", df_stem.shape)
print("\nStates:", df_stem['state'].nunique())
print("Occupations:", df_stem['occupation'].nunique())
print("Races:", df_stem['race'].unique())
print("\nSample:")
print(df_stem.head(10))

✅ Clean STEM dataframe shape: (245, 4)

States: 245
Occupations: 1
Races: ['Hispanic or Latino']

Sample:
                  state                                 occupation  \
0               Alabama  Computer and information systems managers   
1                Alaska  Computer and information systems managers   
2               Arizona  Computer and information systems managers   
3              Arkansas  Computer and information systems managers   
4            California  Computer and information systems managers   
5              Colorado  Computer and information systems managers   
6           Connecticut  Computer and information systems managers   
7              Delaware  Computer and information systems managers   
8  District of Columbia  Computer and information systems managers   
9               Florida  Computer and information systems managers   

                 race  percent  
0  Hispanic or Latino      2.9  
1  Hispanic or Latino      2.1  
2  Hispanic or Latino   

In [ ]:
import pandas as pd
import numpy as np

# ── 1. Load 
df_raw = pd.read_csv(
    '/Users/amanmujawar/Documents/Projects/team4_project/HR-Analysis/Data/ACSEEO5Y2018.EEOALL1R-2026-02-16T043140.csv',
    header=0, low_memory=False
)

# ── 2. Keep only Estimate columns 
estimate_cols = [c for c in df_raw.columns if c.endswith('!!Estimate') or c == 'Label (Grouping)']
df = df_raw[estimate_cols].copy()

# ── 3. Keep only the Total Percent row (row index 2) 
df = df.iloc[[2]].copy()

# ── 4. Melt wide → long 
df_melted = df.melt(id_vars='Label (Grouping)', var_name='column_key', value_name='percent')

# ── 5. Parse column key: split on FIRST two !! only 
# Format: State!!Occupation!!Race(may contain !!)!!Estimate
# We split into max 4 parts
df_melted['column_key_clean'] = df_melted['column_key'].str.replace('!!Estimate$', '', regex=True)

def parse_col(col):
    parts = col.split('!!')
    if len(parts) < 3:
        return pd.Series([None, None, None])
    state = parts[0]
    occupation = parts[1]
    race = ' - '.join(parts[2:])  # rejoin remaining parts as race
    return pd.Series([state, occupation, race])

df_melted[['state', 'occupation', 'race']] = df_melted['column_key_clean'].apply(parse_col)

df_melted['percent'] = (
    df_melted['percent']
    .astype(str)
    .str.replace('%', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df_melted['percent'] = pd.to_numeric(df_melted['percent'], errors='coerce')

df_clean = df_melted.dropna(subset=['percent', 'race', 'occupation']).copy()
df_clean = df_clean[~df_clean['race'].str.contains('Total, race', na=False)]
df_clean = df_clean[~df_clean['race'].str.contains('Balance', na=False)]

def simplify_race(r):
    r = r.strip()
    if 'Hispanic or Latino' in r and 'Not' not in r:
        return 'Hispanic or Latino'
    elif 'White alone' in r:
        return 'White'
    elif 'Black or African American' in r:
        return 'Black or African American'
    elif 'Asian alone' in r:
        return 'Asian'
    elif 'American Indian' in r:
        return 'American Indian / Alaska Native'
    elif 'Native Hawaiian' in r:
        return 'Native Hawaiian / Pacific Islander'
    elif 'Two or more' in r:
        return 'Two or More Races'
    else:
        return None

df_clean['race_clean'] = df_clean['race'].apply(simplify_race)
df_clean = df_clean[df_clean['race_clean'].notna()].copy()

stem_keywords = [
    'engineer', 'computer', 'software', 'mathematician', 'statistician',
    'scientist', 'architect', 'technician', 'drafter', 'surveyor',
    'physicist', 'chemist', 'biologist', 'life scientist', 'physical scientist',
    'information systems', 'network', 'database', 'systems manager'
]
mask = df_clean['occupation'].str.lower().str.contains('|'.join(stem_keywords), na=False)
df_stem = df_clean[mask].copy()

df_stem = df_stem[['state', 'occupation', 'race_clean', 'percent']].rename(
    columns={'race_clean': 'race'}
).reset_index(drop=True)

print("Shape:", df_stem.shape)
print("States:", df_stem['state'].nunique())
print("Occupations:", df_stem['occupation'].nunique())
print("Unique occupations:", df_stem['occupation'].unique().tolist())
print("Races:", df_stem['race'].unique().tolist())
print("\nSample:")
print(df_stem.head(15))

✅ Shape: (1470, 4)
States: 245
Occupations: 1
Unique occupations: ['Computer and information systems managers']
Races: ['Hispanic or Latino', 'White', 'Black or African American', 'American Indian / Alaska Native', 'Asian', 'Native Hawaiian / Pacific Islander']

Sample:
      state                                 occupation  \
0   Alabama  Computer and information systems managers   
1   Alabama  Computer and information systems managers   
2   Alabama  Computer and information systems managers   
3   Alabama  Computer and information systems managers   
4   Alabama  Computer and information systems managers   
5   Alabama  Computer and information systems managers   
6    Alaska  Computer and information systems managers   
7    Alaska  Computer and information systems managers   
8    Alaska  Computer and information systems managers   
9    Alaska  Computer and information systems managers   
10   Alaska  Computer and information systems managers   
11   Alaska  Computer and informa

In [ ]:
df_all_occs = df_clean[['occupation']].drop_duplicates()
print("Total unique occupations:", df_all_occs['occupation'].nunique())
print("\nAll occupations:")
for occ in sorted(df_all_occs['occupation'].dropna().unique()):
    print(" -", occ)

Total unique occupations: 5

All occupations:
 - Administrative services and facilities managers
 - Advertising, marketing, promotions, public relations, and sales managers
 - Computer and information systems managers
 - Financial managers
 - Top executives


In [ ]:
df_final = df_clean[['state', 'occupation', 'race_clean', 'percent']].rename(
    columns={'race_clean': 'race'}
).reset_index(drop=True)


print("Unique states:", df_final['state'].nunique())
print("Unique occupations:", df_final['occupation'].nunique())
print("Unique races:", df_final['race'].nunique())
print("Total rows:", len(df_final))
print("\nSample:")
print(df_final.head(12))

df_final.to_csv('../data/clean_eeo_data.csv', index=False)
print("\n Saved to ../data/clean_eeo_data.csv")

Unique states: 251
Unique occupations: 5
Unique races: 6
Total rows: 7350

Sample:
      state                                         occupation  \
0   Alabama                                     Top executives   
1   Alabama                                     Top executives   
2   Alabama                                     Top executives   
3   Alabama                                     Top executives   
4   Alabama                                     Top executives   
5   Alabama                                     Top executives   
6   Alabama  Advertising, marketing, promotions, public rel...   
7   Alabama  Advertising, marketing, promotions, public rel...   
8   Alabama  Advertising, marketing, promotions, public rel...   
9   Alabama  Advertising, marketing, promotions, public rel...   
10  Alabama  Advertising, marketing, promotions, public rel...   
11  Alabama  Advertising, marketing, promotions, public rel...   

                                  race  percent  
0       